In [2]:
pip install supabase

Note: you may need to restart the kernel to use updated packages.


## Superbase url, key 설정
- 현재 만들어진 superbase api_key, url 넣어서 실행해주세요.

In [9]:
import os

# 1. .streamlit 폴더 만들기 (없으면 생성)
folder_path = ".streamlit"
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"✅ 폴더 생성 완료: {folder_path}")
else:
    print(f"ℹ️ 폴더가 이미 존재합니다: {folder_path}")

# 2. secrets.toml 파일 내용 작성
# 주의: 아래 "https://..."와 "sbp_..." 부분에 본인의 실제 값을 넣고 실행하세요!
toml_content = """
SUPABASE_URL = "url"
SUPABASE_KEY = "key"
"""

file_path = os.path.join(folder_path, "secrets.toml")

# 3. 파일 저장
with open(file_path, "w", encoding="utf-8") as f:
    f.write(toml_content.strip())

print(f"🎉 파일 생성 완료! 위치: {file_path}")
print("이제 터미널에서 'streamlit run main.py'를 실행하면 됩니다.")

ℹ️ 폴더가 이미 존재합니다: .streamlit
🎉 파일 생성 완료! 위치: .streamlit/secrets.toml
이제 터미널에서 'streamlit run main.py'를 실행하면 됩니다.


## 1. 데이터 삽입 테스트
- 홍길동 데이터를 users 테이블에 삽입하는 테스트 코드
- 데이터 삽입 테스트 후 생성되는 더미 데이터를 꼭 테이블에서 삭제

## 2. 로그인 테스트
- 로그인 창 입력 테스트
- 이메일 유효성 검사, db 중복 찾아내는 코드까지 작성

## 회원가입& 기업정보 입력 테스트
- streamlit에서 입력한 데이터가 db에 적재되는 것까지 최종 확인
- 확인할 부분 : 이메일 유효성 검사, 이메일 db 중복확인
- 산업군 분류 : 공공데이터포털 - 소상공인시장진흥공단_상가(상권)정보 업종코드 참조
- 지역분류 : 행정안전부 - 내고장알리미 - 행정구역분류 참조

## 로그인~메인페이지 테스트

In [23]:
%%writefile main.py
import streamlit as st
import uuid
import re
from typing import Dict, Union, Any, Optional
from datetime import date
from supabase import create_client, Client

# 테이블명 정의
# 유효성검사
TABLE_USERS = "users"
TABLE_BUSINESSES = "businesses"
EMAIL_REGEX = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"

# superbase 
def get_supabase_client() -> Client:
    """
    Streamlit Secrets를 사용하여 Supabase 클라이언트를 초기화합니다.

    Streamlit의 secrets.toml 파일에 저장된 설정을 불러와 클라이언트를 생성합니다.
    연결 실패 시 에러 메시지를 출력하고 앱 실행을 중단합니다.

    Raises:
        Exception: 연결 설정이 없거나 초기화 중 오류가 발생할 경우.

    Returns:
        Client: 인증된 Supabase 클라이언트 객체.
    """
    try:
        url: str = st.secrets["SUPABASE_URL"]
        key: str = st.secrets["SUPABASE_KEY"]
        return create_client(url, key)
    except Exception as e:
        st.error(f"Supabase 연결 실패: {e}")
        st.stop()


def check_email_exists(client: Client, email: str) -> bool:
    """
    데이터베이스에 입력된 이메일이 이미 존재하는지 확인합니다.

    Args:
        client (Client): Supabase 클라이언트 인스턴스.
        email (str): 중복 여부를 확인할 사용자 이메일 주소.

    Returns:
        bool: 이메일이 이미 존재하거나 확인 중 에러가 발생하면 True,
              존재하지 않으면 False를 반환합니다.
    """
    try:
        response = client.table(TABLE_USERS).select("user_id").eq("email", email).execute()
        return len(response.data) > 0
    except Exception:
        return True 


def register_new_user(client: Client, user_data: Dict) -> bool:
    """
    새로운 사용자 정보를 'users' 테이블에 저장합니다.

    Args:
        client (Client): Supabase 클라이언트 인스턴스.
        user_data (Dict[str, Any]): 사용자 정보가 담긴 딕셔너리.
                                    (포함 키: user_id, email, password, name, age, gender 등)

    Returns:
        bool: 데이터 저장에 성공하면 True, 실패하면 False를 반환합니다.
    """
    try:
        client.table(TABLE_USERS).insert(user_data).execute()
        return True
    except Exception as e:
        st.error(f"회원가입 오류: {e}")
        return False


def register_business_info(client: Client, biz_data: Dict) -> bool:
    """
    기업(Business) 정보를 'businesses' 테이블에 저장합니다.

    Args:
        client (Client): Supabase 클라이언트 인스턴스.
        biz_data (Dict[str, Any]): 기업 정보가 담긴 딕셔너리.
                                   (포함 키: biz_id, user_id, industry_code, emp_count 등)

    Returns:
        bool: 데이터 저장에 성공하면 True, 실패하면 False를 반환합니다.
    """
    try:
        client.table(TABLE_BUSINESSES).insert(biz_data).execute()
        return True
    except Exception as e:
        st.error(f"기업 정보 저장 오류: {e}")
        return False


def login_user(client: Client, email: str, password: str) -> Optional[Dict]:
    """
    이메일과 비밀번호를 사용하여 사용자 로그인을 처리합니다.

    Args:
        client (Client): Supabase 클라이언트 인스턴스.
        email (str): 사용자 이메일.
        password (str): 사용자 비밀번호.

    Returns:
        Optional[Dict[str, Any]]: 로그인 성공 시 사용자 정보 딕셔너리를 반환하고,
                                  일치하는 정보가 없거나 오류 발생 시 None을 반환합니다.
    """
    try:
        response = client.table(TABLE_USERS).select("*").eq("email", email).eq("password", password).execute()
        if len(response.data) > 0:
            return response.data[0]
        else:
            return None
    except Exception as e:
        st.error(f"로그인 중 오류 발생: {e}")
        return None


def main():
    """
    Streamlit 애플리케이션의 메인 진입점입니다.

    세션 상태(Session State)를 관리하여 페이지 네비게이션(로그인 -> 회원가입 -> 기업정보 -> 완료)을 처리하고,
    각 단계별 UI 렌더링 및 데이터 처리를 수행합니다.
    """
    st.set_page_config(page_title="소상공인 서비스", page_icon="🏢")

    # --- 세션 상태 초기화 ---
    if 'step' not in st.session_state:
        st.session_state.step = 'login' 
    if 'is_logged_in' not in st.session_state:
        st.session_state.is_logged_in = False
    if 'current_user' not in st.session_state:
        st.session_state.current_user = None
    if 'new_user_id' not in st.session_state:
        st.session_state.new_user_id = None
    if 'user_name' not in st.session_state:
        st.session_state.user_name = ""
    
    # [추가됨] 기업 정보 등록 성공 여부를 저장할 상태 변수
    if 'biz_registration_success' not in st.session_state:
        st.session_state.biz_registration_success = False

    # Supabase 클라이언트 연결
    supabase = get_supabase_client()

    # ==========================================
    # 화면 1: 메인 대시보드 (로그인 성공 시)
    # ==========================================
    if st.session_state.is_logged_in:
        user = st.session_state.current_user
        st.title(f"👋 안녕하세요, {user['name']} 사장님!")
        st.success("로그인에 성공하셨습니다.")
        st.write("여기에 서비스의 메인 기능들이 들어갑니다.")
        
        if st.button("로그아웃"):
            st.session_state.is_logged_in = False
            st.session_state.current_user = None
            st.session_state.step = 'login'
            st.rerun()
        return

    # ==========================================
    # 화면 2: 로그인 페이지
    # ==========================================
    if st.session_state.step == 'login':
        st.title("🔐 로그인")
        
        with st.form("login_form"):
            email = st.text_input("이메일", placeholder="example@email.com")
            password = st.text_input("비밀번호", type="password")
            submitted = st.form_submit_button("로그인")
        
        if submitted:
            if not email or not password:
                st.warning("이메일과 비밀번호를 입력해주세요.")
            else:
                user_info = login_user(supabase, email, password)
                if user_info:
                    st.session_state.is_logged_in = True
                    st.session_state.current_user = user_info
                    st.balloons()
                    st.rerun()
                else:
                    st.error("이메일 또는 비밀번호가 일치하지 않습니다.")

        st.markdown("---")
        st.write("아직 계정이 없으신가요?")
        if st.button("회원가입 하러 가기"):
            st.session_state.step = 'signup'
            st.rerun()

    # ==========================================
    # 화면 3: 회원가입 (개인정보)
    # ==========================================
    elif st.session_state.step == 'signup':
        st.title("📝 사장님 회원가입")
        st.info("서비스 이용을 위해 사장님의 정보를 입력해주세요.")

        with st.form("signup_form", clear_on_submit=False):
            col1, col2 = st.columns(2)
            with col1:
                name = st.text_input("이름", placeholder="홍길동")
                age = st.number_input("나이", min_value=1, max_value=120, value=30)
            with col2:
                gender_display = st.selectbox("성별", options=["남성", "여성"])
            
            email = st.text_input("이메일", placeholder="example@email.com")
            password = st.text_input("비밀번호", type="password")
            password_confirm = st.text_input("비밀번호 확인", type="password")

            submit_signup = st.form_submit_button("다음 단계로 (기업 정보 입력)")

        if st.button("로그인 화면으로 돌아가기"):
            st.session_state.step = 'login'
            st.rerun()

        if submit_signup:
            if not name or not email or not password:
                st.warning("모든 필수 항목을 입력해주세요.")
                return
            if not re.match(EMAIL_REGEX, email):
                st.error("이메일 형식이 올바르지 않습니다.")
                return
            if password != password_confirm:
                st.error("비밀번호가 일치하지 않습니다.")
                return
            
            with st.spinner("중복 확인 및 저장 중..."):
                if check_email_exists(supabase, email):
                    st.error("이미 가입된 이메일입니다.")
                    return

                gender_map = {"남성": "M", "여성": "F"}
                new_uuid = str(uuid.uuid4())

                user_data = {
                    "user_id": new_uuid,
                    "email": email,
                    "password": password, 
                    "name": name,
                    "age": age,
                    "gender": gender_map[gender_display]
                }

                if register_new_user(supabase, user_data):
                    st.session_state.new_user_id = new_uuid
                    st.session_state.user_name = name
                    st.session_state.step = 'business_info'
                    st.rerun()

    # ==========================================
    # 화면 4: 기업 정보 입력
    # ==========================================
    elif st.session_state.step == 'business_info':
        
        # [수정] 이미 성공했다면 성공 화면만 보여줌
        if st.session_state.biz_registration_success:
            st.title("🎉 가입 완료!")
            st.balloons()
            st.success("모든 가입 절차가 완료되었습니다.")
            
            if st.button("로그인 화면으로 이동"):
                st.session_state.step = 'login'
                st.session_state.new_user_id = None
                st.session_state.biz_registration_success = False # 상태 초기화
                st.rerun()
            return  # 아래 폼 렌더링을 건너뜀

        # --- 아직 제출 안 했을 때 폼 보여주기 ---
        st.title("🏢 기업 정보 입력")
        st.success(f"환영합니다, {st.session_state.user_name} 사장님! 사업장 정보를 입력해주세요.")
        
        with st.form("business_form"):
            industry = st.selectbox(
                "업종 (Industry)", 
                options=[
                    "소매업", "숙박업", "음식점업", "부동산업", "전문, 과학 및 기술 서비스업", 
                    "사업시설 관리, 사업 지원 및 임대 서비스업", "교육 서비스업", "보건의료업", 
                    "예술, 스포츠 및 여가관련 서비스업", "수리 및 개인 서비스업"
                ],
                help="가장 가까운 업종을 선택해주세요."
            )
            
            region = st.selectbox(
                "지역 (Region)",
                options=[
                    "서울특별시", "경기도", "인천광역시", "부산광역시", "대구광역시", 
                    "광주광역시", "대전광역시", "울산광역시", "세종특별자치시", "충청북도", 
                    "충청남도", "전라남도", "경상북도", "경상남도", "강원특별자치도", 
                    "전북특별자치도", "제주특별자치도"
                ]
            )

            col1, col2 = st.columns(2)
            with col1:
                emp_count = st.number_input("직원 수 (명)", min_value=0, value=0)
            with col2:
                opening_date = st.date_input("개업일", value=date.today())

            submit_biz = st.form_submit_button("가입 완료하기")

        if submit_biz:
            biz_data = {
                "biz_id": str(uuid.uuid4()),
                "user_id": st.session_state.new_user_id,
                "industry_code": industry,
                "emp_count": emp_count,
                "opening_date": str(opening_date),
                "region_code": region
            }

            with st.spinner("최종 등록 중..."):
                if register_business_info(supabase, biz_data):
                    # 성공 상태를 저장하고 화면 리로드
                    st.session_state.biz_registration_success = True
                    st.rerun()

if __name__ == "__main__":
    main()

Overwriting main.py
